<a href="https://colab.research.google.com/github/kukanmani06/Hiver_SDE_Assignment/blob/main/Hiver_SDE_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup and dataset loading



In [ ]:
import os
import re
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path

RANDOM_STATE = 42
BRAND = "AmazonHelp"
BASE_DIR = Path("/content/hiver_dataset")
CSV_PATH = BASE_DIR / "twcs" / "twcs.csv"

# If the dataset is not already present, upload the zip in Colab.
if not CSV_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    zip_path = next(iter(uploaded))
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(BASE_DIR)

df = pd.read_csv(CSV_PATH)
print("Full dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


Saving archive (6).zip to archive (6).zip
Full dataset shape: (2811774, 7)
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


## 2. Select the brand and construct customer → historical reply pairs


In [ ]:
brands = df.loc[df["inbound"] == False, "author_id"].value_counts()
print("Top brands:")
display(brands.head(10).to_frame("brand_replies"))

amazon = df[
    (df["author_id"] == BRAND) |
    (df["text"].astype(str).str.contains(r"@AmazonHelp", case=False, na=False))
].copy()

customer_df = amazon[amazon["inbound"] == True].copy()
company_df = amazon[amazon["author_id"] == BRAND].copy()

company_lookup = company_df.set_index("tweet_id")["text"].to_dict()

def first_id(value):
    if pd.isna(value):
        return np.nan
    m = re.search(r"\d+", str(value))
    return float(m.group()) if m else np.nan

customer_df["reply_id"] = customer_df["response_tweet_id"].apply(first_id)
customer_df["historical_reply"] = customer_df["reply_id"].map(company_lookup)

resolved_df = customer_df[customer_df["historical_reply"].notna()].copy()
resolved_df = resolved_df.drop_duplicates(subset=["tweet_id"]).reset_index(drop=True)
resolved_df["analysis_text"] = (
    resolved_df["text"].fillna("")
    .str.replace(r"@AmazonHelp", "", regex=True, case=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("AmazonHelp customer tweets:", len(customer_df))
print("Customer tweets with linked historical reply:", len(resolved_df))
display(resolved_df[["tweet_id","text","historical_reply"]].head(10))


Top brands:


,brand_replies
author_id,
AmazonHelp,169840
AppleSupport,106860
Uber_Support,56270
SpotifyCares,43265
Delta,42253
Tesco,38573
AmericanAir,36764
TMobileHelp,34317
comcastcares,33031


AmazonHelp customer tweets: 135160
Customer tweets with linked historical reply: 91256


,tweet_id,text,historical_reply
0,271,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
1,274,@AmazonHelp こちらこそありがとうございました。,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
2,616,@AmazonHelp 3 different people have given 3 di...,@115820 We'd like to take a further look into ...
3,623,"@AmazonHelp Okay, danke für die Info",@115824 Wir haben zu danken. Schönen Abend noc...
4,627,@AmazonHelp @115826 Yeah this is crazy we’re l...,@115827 Thanks for your patience. ^KM
5,634,@115821 @AmazonHelp why is my order at my loca...,@115831 I'm sorry for the wait. Please reach o...
6,638,@AmazonHelp Hi ready for some help,@115834 Were you able to reach us at the link ...
7,640,@AmazonHelp Is the Echo Show no longer supported?,"@115834 The Echo Show is supported, please rea..."
8,643,Bought an @115821 Echo Show and it won’t recog...,"@115834 Oh no, I'm sorry for the issues! For t..."
9,645,@AmazonHelp That page is useless - doesn’t all...,@115835 You can also request a call back here ...


In [ ]:
INTENTS = [
    "delivery",
    "order",
    "refund_payment",
    "return",
    "product_issue",
    "prime",
    "account_login",
    "customer_service",
    "other",
]

# Canonical labels used everywhere in this notebook.
# Legacy files may contain `product_return`; it is treated only as an alias for `return`.
LEGACY_INTENT_ALIASES = {"product_return": "return"}


In [ ]:
model_df = resolved_df.sample(n=min(20000, len(resolved_df)), random_state=RANDOM_STATE).reset_index(drop=True)
print("Model sample:", model_df.shape)


Model sample: (20000, 10)


## 3. Baseline 1 — majority class

This is the required trivial baseline. It predicts the most common intent in the training data.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

# --- FIX: Generate silver_intent column for model_df ---
def assign_silver_intent(text):
    text = str(text).lower()
    if "delivery" in text or "shipped" in text or "shipping" in text:
        return "delivery"
    if "order" in text or "purchase" in text:
        return "order"
    if "refund" in text or "payment" in text or "charged" in text:
        return "refund_payment"
    if "return" in text:
        return "return"
    if "product" in text or "item" in text or "damaged" in text or "faulty" in text:
        return "product_issue"
    if "prime" in text:
        return "prime"
    if "account" in text or "login" in text or "password" in text:
        return "account_login"
    if "customer service" in text or "support" in text or "help" in text:
        return "customer_service"
    return "other"

model_df["silver_intent"] = model_df["analysis_text"].apply(assign_silver_intent)
# --------------------------------------------------------

train_df, test_df = train_test_split(
    model_df, test_size=0.20, random_state=RANDOM_STATE, stratify=model_df["silver_intent"]
)
majority_intent = train_df["silver_intent"].mode()[0]
majority_pred = np.repeat(majority_intent, len(test_df))

majority_acc = accuracy_score(test_df["silver_intent"], majority_pred)
majority_f1 = f1_score(test_df["silver_intent"], majority_pred, average="macro", zero_division=0)
print("Majority baseline intent:", majority_intent)
print("Accuracy:", round(majority_acc, 4))
print("Macro F1:", round(majority_f1, 4))

Majority baseline intent: other
Accuracy: 0.5833
Macro F1: 0.0819


## 4. Baseline 2 — TF-IDF + Logistic Regression

This replaces the original notebook's multiple classifier experiments with one clear, reproducible simple baseline.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

clf = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, max_features=20000, ngram_range=(1,2), min_df=2)),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

clf.fit(train_df["analysis_text"], train_df["silver_intent"])
test_pred = clf.predict(test_df["analysis_text"])

tfidf_acc = accuracy_score(test_df["silver_intent"], test_pred)
tfidf_f1 = f1_score(test_df["silver_intent"], test_pred, average="macro", zero_division=0)
print("TF-IDF + Logistic Regression")
print("Accuracy:", round(tfidf_acc, 4))
print("Macro F1:", round(tfidf_f1, 4))
print(classification_report(test_df["silver_intent"], test_pred, zero_division=0))


TF-IDF + Logistic Regression
Accuracy: 0.9233
Macro F1: 0.8596
                  precision    recall  f1-score   support

   account_login       0.83      0.92      0.88       104
customer_service       0.73      0.92      0.81       216
        delivery       0.97      0.89      0.93       421
           order       0.92      0.84      0.88       455
           other       0.98      0.96      0.97      2333
           prime       0.84      0.86      0.85       138
   product_issue       0.76      0.86      0.80       151
  refund_payment       0.75      0.80      0.78       123
          return       0.77      0.92      0.84        59

        accuracy                           0.92      4000
       macro avg       0.84      0.89      0.86      4000
    weighted avg       0.93      0.92      0.92      4000



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

retrieval_data = train_df[["tweet_id", "analysis_text", "historical_reply", "silver_intent"]].copy()
retrieval_data = retrieval_data[retrieval_data["historical_reply"].astype(str).str.len() > 10].reset_index(drop=True)

reply_vectorizer = TfidfVectorizer(lowercase=True, max_features=20000, ngram_range=(1,2), min_df=2)
reply_matrix = reply_vectorizer.fit_transform(retrieval_data["analysis_text"])

def clean_reply(reply):
    reply = str(reply)
    reply = re.sub(r"@\d+", "", reply)
    reply = re.sub(r"@AmazonHelp", "", reply, flags=re.I)
    reply = re.sub(r"https?://\S+", "", reply)
    reply = re.sub(r"\^[A-Z]{1,3}\b", "", reply)
    return re.sub(r"\s+", " ", reply).strip()

def retrieve_reply(customer_message):
    query = reply_vectorizer.transform([customer_message])
    scores = cosine_similarity(query, reply_matrix).ravel()
    best = int(scores.argmax())
    return {
        "matched_tweet_id": int(retrieval_data.iloc[best]["tweet_id"]),
        "matched_customer_message": retrieval_data.iloc[best]["analysis_text"],
        "matched_reply": clean_reply(retrieval_data.iloc[best]["historical_reply"]),
        "matched_intent": retrieval_data.iloc[best]["silver_intent"],
        "similarity_score": float(scores[best]),
    }


## 5. Final agent — one safety policy



In [ ]:
SUPPORTED_AUTO_INTENTS = {"delivery", "order", "prime", "return", "product_issue"}
SENSITIVE_INTENTS = {"refund_payment", "account_login", "customer_service"}
SIMILARITY_THRESHOLD = 0.55

def is_vague(text):
    t = str(text).lower().strip()
    if len(t.split()) <= 5:
        return True
    vague = ["yes", "okay", "ok", "thanks", "thank you", "any update", "still waiting"]
    return any(t == x or t.startswith(x + " ") for x in vague)

def bad_reply(reply):
    t = str(reply).lower().strip()
    if len(t) < 15:
        return True
    bad = ["1/2", "2/2", "3/3", "here:"]
    return any(x in t for x in bad)

def decide_action(predicted_intent, retrieved_intent, similarity_score, customer_message, reply):
    reasons = []
    if predicted_intent not in SUPPORTED_AUTO_INTENTS:
        reasons.append("Intent is unclear or sensitive")
    if similarity_score < SIMILARITY_THRESHOLD:
        reasons.append("Historical similarity is below safety threshold")
    if predicted_intent != retrieved_intent:
        reasons.append("Intent mismatch with historical example")
    if is_vague(customer_message):
        reasons.append("Customer message is too short or vague")
    if bad_reply(reply):
        reasons.append("Retrieved historical reply is incomplete or unsafe")
    if reasons:
        return "ESCALATE", "; ".join(reasons)
    return "AUTO_HANDLE", "Supported intent, strong historical match, and clear customer message"

def support_agent(customer_message):
    cleaned = re.sub(r"@AmazonHelp", "", str(customer_message), flags=re.I).strip()
    predicted = clf.predict([cleaned])[0]
    retrieved = retrieve_reply(cleaned)
    decision, reason = decide_action(
        predicted, retrieved["matched_intent"], retrieved["similarity_score"], cleaned, retrieved["matched_reply"]
    )
    if decision == "AUTO_HANDLE":
        final_reply = retrieved["matched_reply"]
    else:
        final_reply = "Thank you for contacting AmazonHelp. Your request needs further assistance from our support team. A support agent will review your issue and help you shortly."
    return {
        "intent": predicted,
        "similarity_score": round(retrieved["similarity_score"], 4),
        "decision": decision,
        "reason": reason,
        "final_reply": final_reply,
        "evidence_reply": retrieved["matched_reply"],
        "evidence_tweet_id": retrieved["matched_tweet_id"],
    }

test_message = "My package has not arrived and the tracking has not updated."
print(support_agent(test_message))


{'intent': 'other', 'similarity_score': 0.5589, 'decision': 'ESCALATE', 'reason': 'Intent is unclear or sensitive; Retrieved historical reply is incomplete or unsafe', 'final_reply': 'Thank you for contacting AmazonHelp. Your request needs further assistance from our support team. A support agent will review your issue and help you shortly.', 'evidence_reply': "Oh no! Let's take a look at this with you in real time. Please click here: to see what's going on", 'evidence_tweet_id': 307199}


6. Golden evaluation set —



In [ ]:
# Sample a balanced-ish golden queue from the untouched resolved dataset.
used_ids = set(train_df["tweet_id"]).union(set(test_df["tweet_id"]))
candidate = resolved_df[~resolved_df["tweet_id"].isin(used_ids)].copy()

# Prefer messages with enough context and a linked historical reply.
candidate = candidate[candidate["analysis_text"].str.split().str.len() >= 4].copy()
golden = candidate.sample(n=min(200, len(candidate)), random_state=123).copy()

golden["human_intent"] = ""
golden["human_reply_quality"] = ""
golden["human_action"] = ""
golden["human_notes"] = ""

golden_columns = ["tweet_id", "analysis_text", "historical_reply", "human_intent", "human_reply_quality", "human_action", "human_notes"]
golden = golden[golden_columns].reset_index(drop=True)

annotation_path = "/content/amazonhelp_golden_200_to_label.csv"
golden.to_csv(annotation_path, index=False)
print("Golden annotation file:", annotation_path)
print("Rows:", len(golden))
display(golden.head(10))


Golden annotation file: /content/amazonhelp_golden_200_to_label.csv
Rows: 200


,tweet_id,analysis_text,historical_reply,human_intent,human_reply_quality,human_action,human_notes
0,281979,Your product wich is in pic looks diffrent nd ...,@183175 Apologies for the inconvenience. Reque...,,,,
1,671655,I don't have an Amazon account with that email...,"@280258 Hey, you can set up an Amazon account ...",,,,
2,40422,But please arrange return pick up from your hand,"@120261 Please don't worry, our team will get ...",,,,
3,1665280,"C'est inexcusable, je dois faire un transfert ...","@507413 Nous n'avons pas accès à votre compte,...",,,,
4,946601,I am very disappointing with the service of se...,@344469 Kindly share your details here: https:...,,,,
5,73532,No i ordered one item and it says it has been ...,@132441 Thanks for the additional information!...,,,,
6,318264,It seems to be cooperating now. Its preference...,@191939 Delighted it is working now.^CD,,,,
7,472347,I normally buy a few bits on a Friday but its ...,@227489 This is not the experience we want you...,,,,
8,508132,@236664 I ordered something three different it...,@249440 I'm sorry to hear about the delivery t...,,,,
9,413015,Gibts da ne Vorlage ? Ich kann nicht schlafen ...,@213578 Schau mal hier: https://t.co/7M84KcSxS...,,,,


### Load the completed human-labelled golden set

Run this section only after you have manually filled the CSV.

In [ ]:
import pandas as pd

# CSV load
df = pd.read_csv("amazonhelp_golden_200_to_labelled.csv")

# Basic check
print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

# Missing values
print("\nMissing values:")
print(df.isnull().sum())

# Human label columns
for col in ["human_intent", "human_reply_quality", "human_action"]:
    if col in df.columns:
        print(f"\n{col} value counts:")
        print(df[col].value_counts(dropna=False))

# Duplicate customer messages
if "analysis_text" in df.columns:
    print("\nDuplicate customer messages:",
          df["analysis_text"].duplicated().sum())

# Show first 10 rows
display(df.head(10))

Rows: 200
Columns:
['tweet_id', 'analysis_text', 'historical_reply', 'human_intent', 'human_reply_quality', 'human_action', 'human_notes']

Missing values:
tweet_id               0
analysis_text          0
historical_reply       0
human_intent           0
human_reply_quality    0
human_action           0
human_notes            0
dtype: int64

human_intent value counts:
human_intent
delivery            52
customer_service    48
product_issue       29
other               15
refund_payment      15
order               13
return              11
prime               10
account_login        7
Name: count, dtype: int64

human_reply_quality value counts:
human_reply_quality
Acceptable    79
Poor          73
Good          48
Name: count, dtype: int64

human_action value counts:
human_action
ESCALATE       150
AUTO_HANDLE     50
Name: count, dtype: int64

Duplicate customer messages: 0


,tweet_id,analysis_text,historical_reply,human_intent,human_reply_quality,human_action,human_notes
0,281979,Your product wich is in pic looks diffrent nd ...,@183175 Apologies for the inconvenience. Reque...,product_issue,Acceptable,ESCALATE,Product received differs from pictured product
1,671655,I don't have an Amazon account with that email...,"@280258 Hey, you can set up an Amazon account ...",account_login,Good,AUTO_HANDLE,Customer asks about creating an Amazon account...
2,40422,But please arrange return pick up from your hand,"@120261 Please don't worry, our team will get ...",return,Acceptable,ESCALATE,Customer requests return pickup; support needs...
3,1665280,"C'est inexcusable, je dois faire un transfert ...","@507413 Nous n'avons pas accès à votre compte,...",customer_service,Acceptable,ESCALATE,Customer needs support for a time-sensitive co...
4,946601,I am very disappointing with the service of se...,@344469 Kindly share your details here: https:...,customer_service,Acceptable,ESCALATE,Seller support case has remained unresolved fo...
5,73532,No i ordered one item and it says it has been ...,@132441 Thanks for the additional information!...,delivery,Acceptable,ESCALATE,Order is marked delivered but part of the item...
6,318264,It seems to be cooperating now. Its preference...,@191939 Delighted it is working now.^CD,product_issue,Good,AUTO_HANDLE,Customer indicates the issue is now working no...
7,472347,I normally buy a few bits on a Friday but its ...,@227489 This is not the experience we want you...,delivery,Acceptable,AUTO_HANDLE,Customer reports unexpected delivery-date chan...
8,508132,@236664 I ordered something three different it...,@249440 I'm sorry to hear about the delivery t...,delivery,Acceptable,ESCALATE,Customer reports failed delivery attempts to a...
9,413015,Gibts da ne Vorlage ? Ich kann nicht schlafen ...,@213578 Schau mal hier: https://t.co/7M84KcSxS...,product_issue,Good,AUTO_HANDLE,Customer asks for a product template/reference...


In [ ]:
# Change this path if you uploaded the completed annotation file under another name.
# For the final submission, use the CSV that you have genuinely reviewed.
GOLDEN_PATH = "/content/amazonhelp_golden_200_to_labelled.csv"
golden_eval = pd.read_csv(GOLDEN_PATH)

required = {"tweet_id","analysis_text","human_intent","human_reply_quality","human_action"}
missing = required - set(golden_eval.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Normalize one legacy label if it exists in an older annotation file.
golden_eval["human_intent"] = golden_eval["human_intent"].replace(LEGACY_INTENT_ALIASES)

invalid_intents = sorted(set(golden_eval["human_intent"].dropna().astype(str)) - set(INTENTS))
if invalid_intents:
    raise ValueError(f"Invalid human_intent labels: {invalid_intents}. Use only: {INTENTS}")

for col in ["human_intent", "human_reply_quality", "human_action"]:
    if (golden_eval[col].fillna("").astype(str).str.strip() == "").any():
        raise ValueError(f"Golden set still contains blank {col} labels.")

valid_quality = {"Good", "Acceptable", "Poor"}
valid_actions = {"AUTO_HANDLE", "ESCALATE"}
if not set(golden_eval["human_reply_quality"]).issubset(valid_quality):
    raise ValueError("human_reply_quality must be Good, Acceptable, or Poor.")
if not set(golden_eval["human_action"]).issubset(valid_actions):
    raise ValueError("human_action must be AUTO_HANDLE or ESCALATE.")

results = []
for _, row in golden_eval.iterrows():
    r = support_agent(row["analysis_text"])
    results.append(r)

pred = pd.DataFrame(results)
golden_scored = pd.concat([golden_eval.reset_index(drop=True), pred.reset_index(drop=True)], axis=1)

intent_acc = accuracy_score(golden_scored["human_intent"], golden_scored["intent"])
intent_f1 = f1_score(golden_scored["human_intent"], golden_scored["intent"], average="macro", zero_division=0)

print("Golden rows:", len(golden_scored))
print("Golden intent accuracy:", round(intent_acc, 4))
print("Golden intent macro F1:", round(intent_f1, 4))
print("Auto-handle rate:", round((golden_scored["decision"] == "AUTO_HANDLE").mean(), 4))
print("Escalation rate:", round((golden_scored["decision"] == "ESCALATE").mean(), 4))
print("\nIntent report:")
print(classification_report(golden_scored["human_intent"], golden_scored["intent"], labels=INTENTS, zero_division=0))


Golden rows: 200
Golden intent accuracy: 0.27
Golden intent macro F1: 0.2896
Auto-handle rate: 0.0
Escalation rate: 1.0

Intent report:
                  precision    recall  f1-score   support

        delivery       0.55      0.31      0.40        52
           order       0.29      0.46      0.35        13
  refund_payment       0.50      0.27      0.35        15
          return       0.75      0.27      0.40        11
   product_issue       0.50      0.10      0.17        29
           prime       0.38      0.30      0.33        10
   account_login       0.50      0.14      0.22         7
customer_service       0.33      0.12      0.18        48
           other       0.12      0.80      0.20        15

        accuracy                           0.27       200
       macro avg       0.43      0.31      0.29       200
    weighted avg       0.44      0.27      0.28       200



## 9. Compare the agent against the two baselines on the same golden set



In [ ]:
majority_golden = np.repeat(majority_intent, len(golden_scored))
tfidf_golden = clf.predict(golden_scored["analysis_text"])

baseline_comparison = pd.DataFrame([
    {"System": "Majority baseline", "Accuracy": accuracy_score(golden_scored["human_intent"], majority_golden), "Macro_F1": f1_score(golden_scored["human_intent"], majority_golden, average="macro", zero_division=0)},
    {"System": "TF-IDF + Logistic Regression", "Accuracy": accuracy_score(golden_scored["human_intent"], tfidf_golden), "Macro_F1": f1_score(golden_scored["human_intent"], tfidf_golden, average="macro", zero_division=0)},
    {"System": "Final Support Agent", "Accuracy": intent_acc, "Macro_F1": intent_f1},
])
baseline_comparison[["Accuracy","Macro_F1"]] = baseline_comparison[["Accuracy","Macro_F1"]].round(4)
display(baseline_comparison)


,System,Accuracy,Macro_F1
0,Majority baseline,0.075,0.0155
1,TF-IDF + Logistic Regression,0.270,0.2896
2,Final Support Agent,0.270,0.2896


In [ ]:
quality_map = {"Good": 2, "Acceptable": 1, "Poor": 0}
golden_scored["quality_score"] = golden_scored["human_reply_quality"].map(quality_map)

print("Mean historical reply quality (human):", round(golden_scored["quality_score"].mean(), 3))
print("Human action distribution:")
display(golden_scored["human_action"].value_counts().to_frame("count"))

# Decision agreement is a simple operational metric.
action_acc = accuracy_score(golden_scored["human_action"], golden_scored["decision"])
print("Agent vs human action agreement:", round(action_acc, 4))


Mean historical reply quality (human): 0.875
Human action distribution:


,count
human_action,
ESCALATE,150
AUTO_HANDLE,50


Agent vs human action agreement: 0.75


In [ ]:
# Free Open-Source LLM Judge
# Using FLAN-T5 directly (correct method)

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print("Loading free open-source LLM judge...")

MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("✅ Judge model loaded successfully!")
print("Model:", MODEL_NAME)
print("Device:", device)

Loading free open-source LLM judge...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Judge model loaded successfully!
Model: google/flan-t5-small
Device: cpu


In [ ]:
# ============================================================
# CELL 23 — IMPROVED LLM-AS-JUDGE
# ============================================================

import pandas as pd
from tqdm.auto import tqdm

# ------------------------------------------------------------
# LLM Judge Function
# ------------------------------------------------------------

def judge_reply_quality(customer_message, agent_reply):

    prompt = f"""
You are grading an AI customer-support reply.

CUSTOMER:
{customer_message}

AI REPLY:
{agent_reply}

Choose the correct score:

3 = GOOD
- Directly addresses the customer's problem
- Helpful and relevant
- Gives an appropriate next step or useful information
- Clear and professional

2 = ACCEPTABLE
- Partially addresses the problem
- Relevant but incomplete
- Useful but could be improved

1 = POOR
- Does not address the customer's problem
- Irrelevant, incorrect, or unhelpful
- Gives no useful resolution or next step

IMPORTANT:
Judge the AI REPLY itself.
Do not judge the writing style alone.
Do not assume the reply is good just because it is polite.

Return ONLY ONE NUMBER:
3
2
1
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    outputs = model.generate(
        **inputs,
        max_new_tokens=3,
        do_sample=False
    )

    result = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    # --------------------------------------------------------
    # Convert numeric answer to quality label
    # --------------------------------------------------------

    if result.startswith("3"):
        return "Good"

    elif result.startswith("2"):
        return "Acceptable"

    elif result.startswith("1"):
        return "Poor"

    else:
        # Try to detect a number anywhere in the response
        if "3" in result:
            return "Good"
        elif "2" in result:
            return "Acceptable"
        elif "1" in result:
            return "Poor"
        else:
            return "Unknown"


# ============================================================
# RUN JUDGE ON 50 GOLDEN EXAMPLES
# ============================================================

JUDGE_N = min(50, len(golden_scored))

judge_results = []

print("=" * 60)
print(f"Running improved LLM judge on {JUDGE_N} examples...")
print("=" * 60)
print()


for idx in tqdm(range(JUDGE_N)):

    # Customer message
    customer_message = str(
        golden_scored.iloc[idx]["analysis_text"]
    )

    # IMPORTANT:
    # This is the ACTUAL chatbot-generated reply.
    # We are NOT using historical_reply.
    agent_reply = str(
        golden_scored.iloc[idx]["final_reply"]
    )

    # Human evaluation
    human_quality = str(
        golden_scored.iloc[idx]["human_reply_quality"]
    )

    # LLM evaluation
    llm_quality = judge_reply_quality(
        customer_message,
        agent_reply
    )

    judge_results.append({
        "tweet_id": golden_scored.iloc[idx]["tweet_id"],
        "customer_message": customer_message,
        "agent_reply": agent_reply,
        "human_reply_quality": human_quality,
        "llm_judge_quality": llm_quality
    })


# ============================================================
# CREATE JUDGE AUDIT TABLE
# ============================================================

judge_audit = pd.DataFrame(judge_results)


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 60)
print("✅ IMPROVED LLM JUDGE COMPLETED")
print("=" * 60)

print("\nNumber of examples judged:", len(judge_audit))


# LLM distribution
print("\nLLM Judge Distribution:")
print(
    judge_audit["llm_judge_quality"]
    .value_counts(dropna=False)
)


# Human distribution
print("\nHuman Reply Quality Distribution:")
print(
    judge_audit["human_reply_quality"]
    .value_counts(dropna=False)
)


# Unknown count
unknown_count = (
    judge_audit["llm_judge_quality"] == "Unknown"
).sum()

print("\nUnknown predictions:", unknown_count)


# ============================================================
# SHOW FIRST 10 RESULTS
# ============================================================

print("\nFirst 10 Judge Results:")

display(
    judge_audit[
        [
            "tweet_id",
            "customer_message",
            "agent_reply",
            "human_reply_quality",
            "llm_judge_quality"
        ]
    ].head(10)
)

Running improved LLM judge on 50 examples...



  0%|          | 0/50 [00:00<?, ?it/s]


✅ IMPROVED LLM JUDGE COMPLETED

Number of examples judged: 50

LLM Judge Distribution:
llm_judge_quality
Poor    29
Good    21
Name: count, dtype: int64

Human Reply Quality Distribution:
human_reply_quality
Acceptable    25
Good          13
Poor          12
Name: count, dtype: int64

Unknown predictions: 0

First 10 Judge Results:


,tweet_id,customer_message,agent_reply,human_reply_quality,llm_judge_quality
0,281979,Your product wich is in pic looks diffrent nd ...,Thank you for contacting AmazonHelp. Your requ...,Acceptable,Poor
1,671655,I don't have an Amazon account with that email...,Thank you for contacting AmazonHelp. Your requ...,Good,Good
2,40422,But please arrange return pick up from your hand,Thank you for contacting AmazonHelp. Your requ...,Acceptable,Good
3,1665280,"C'est inexcusable, je dois faire un transfert ...",Thank you for contacting AmazonHelp. Your requ...,Acceptable,Poor
4,946601,I am very disappointing with the service of se...,Thank you for contacting AmazonHelp. Your requ...,Acceptable,Good
5,73532,No i ordered one item and it says it has been ...,Thank you for contacting AmazonHelp. Your requ...,Acceptable,Poor
6,318264,It seems to be cooperating now. Its preference...,Thank you for contacting AmazonHelp. Your requ...,Good,Poor
7,472347,I normally buy a few bits on a Friday but its ...,Thank you for contacting AmazonHelp. Your requ...,Acceptable,Poor
8,508132,@236664 I ordered something three different it...,Thank you for contacting AmazonHelp. Your requ...,Acceptable,Poor
9,413015,Gibts da ne Vorlage ? Ich kann nicht schlafen ...,Thank you for contacting AmazonHelp. Your requ...,Good,Good


In [ ]:
# ============================================================
# CELL 24 — HUMAN vs LLM JUDGE AGREEMENT
# ============================================================

from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
import pandas as pd

# ------------------------------------------------------------
# Valid quality labels
# ------------------------------------------------------------

VALID_LABELS = [
    "Good",
    "Acceptable",
    "Poor"
]

# ------------------------------------------------------------
# Keep only valid LLM + human labels
# ------------------------------------------------------------

agreement_df = judge_audit[
    judge_audit["human_reply_quality"].isin(VALID_LABELS)
    &
    judge_audit["llm_judge_quality"].isin(VALID_LABELS)
].copy()

print("=" * 60)
print("HUMAN vs LLM JUDGE AGREEMENT")
print("=" * 60)

print("\nExamples used:", len(agreement_df))


# ------------------------------------------------------------
# Exact agreement
# ------------------------------------------------------------

exact_agreement = accuracy_score(
    agreement_df["human_reply_quality"],
    agreement_df["llm_judge_quality"]
)

print(
    f"\nHuman-LLM Exact Agreement: "
    f"{exact_agreement:.2%}"
)


# ------------------------------------------------------------
# Cohen's Kappa
# ------------------------------------------------------------

kappa = cohen_kappa_score(
    agreement_df["human_reply_quality"],
    agreement_df["llm_judge_quality"],
    labels=VALID_LABELS
)

print(
    f"Cohen's Kappa: {kappa:.4f}"
)


# ------------------------------------------------------------
# Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    agreement_df["human_reply_quality"],
    agreement_df["llm_judge_quality"],
    labels=VALID_LABELS
)

confusion_df = pd.DataFrame(
    cm,
    index=[
        "Human Good",
        "Human Acceptable",
        "Human Poor"
    ],
    columns=[
        "LLM Good",
        "LLM Acceptable",
        "LLM Poor"
    ]
)

print("\nConfusion Matrix:")
display(confusion_df)


# ------------------------------------------------------------
# Agreement / disagreement count
# ------------------------------------------------------------

agreement_df["exact_match"] = (
    agreement_df["human_reply_quality"]
    ==
    agreement_df["llm_judge_quality"]
)

agreements = agreement_df["exact_match"].sum()
disagreements = len(agreement_df) - agreements

print("\nExact agreements:", agreements)
print("Disagreements:", disagreements)


# ------------------------------------------------------------
# Human distribution
# ------------------------------------------------------------

print("\nHuman Label Distribution:")
print(
    agreement_df["human_reply_quality"]
    .value_counts()
)


# ------------------------------------------------------------
# LLM distribution
# ------------------------------------------------------------

print("\nLLM Judge Label Distribution:")
print(
    agreement_df["llm_judge_quality"]
    .value_counts()
)


# ------------------------------------------------------------
# Show disagreements
# ------------------------------------------------------------

print("\nSample Disagreements:")

display(
    agreement_df[
        agreement_df["exact_match"] == False
    ][
        [
            "tweet_id",
            "customer_message",
            "agent_reply",
            "human_reply_quality",
            "llm_judge_quality"
        ]
    ].head(10)
)

HUMAN vs LLM JUDGE AGREEMENT

Examples used: 50

Human-LLM Exact Agreement: 26.00%
Cohen's Kappa: 0.0154

Confusion Matrix:


,LLM Good,LLM Acceptable,LLM Poor
Human Good,7,0,6
Human Acceptable,8,0,17
Human Poor,6,0,6



Exact agreements: 13
Disagreements: 37

Human Label Distribution:
human_reply_quality
Acceptable    25
Good          13
Poor          12
Name: count, dtype: int64

LLM Judge Label Distribution:
llm_judge_quality
Poor    29
Good    21
Name: count, dtype: int64

Sample Disagreements:


,tweet_id,customer_message,agent_reply,human_reply_quality,llm_judge_quality
0,281979,Your product wich is in pic looks diffrent nd ...,Thank you for contacting AmazonHelp. Your requ...,Acceptable,Poor
2,40422,But please arrange return pick up from your hand,Thank you for contacting AmazonHelp. Your requ...,Acceptable,Good
3,1665280,"C'est inexcusable, je dois faire un transfert ...",Thank you for contacting AmazonHelp. Your requ...,Acceptable,Poor
4,946601,I am very disappointing with the service of se...,Thank you for contacting AmazonHelp. Your requ...,Acceptable,Good
5,73532,No i ordered one item and it says it has been ...,Thank you for contacting AmazonHelp. Your requ...,Acceptable,Poor
6,318264,It seems to be cooperating now. Its preference...,Thank you for contacting AmazonHelp. Your requ...,Good,Poor
7,472347,I normally buy a few bits on a Friday but its ...,Thank you for contacting AmazonHelp. Your requ...,Acceptable,Poor
8,508132,@236664 I ordered something three different it...,Thank you for contacting AmazonHelp. Your requ...,Acceptable,Poor
10,312965,No info since yesterday. Really 😒 https://t.co...,Thank you for contacting AmazonHelp. Your requ...,Acceptable,Good
11,179060,16th of September with Amazon. Received the co...,Thank you for contacting AmazonHelp. Your requ...,Poor,Good


In [ ]:
# ============================================================
# LLM-AS-A-JUDGE LIMITATION ANALYSIS
# ============================================================

print("LLM-as-a-Judge Limitation Analysis")
print("=" * 60)

print(f"\nJudge audit size: {len(judge_audit)}")
print(f"Human-LLM exact agreement: {exact_agreement:.2%}")
print(f"Cohen's kappa: {kappa:.4f}")

judge_distribution = judge_audit["llm_judge_quality"].value_counts()

print("\nLLM Judge distribution:")
print(judge_distribution)

human_distribution = judge_audit["human_reply_quality"].value_counts()

print("\nHuman distribution:")
print(human_distribution)

# Check for label collapse
missing_labels = [
    label for label in VALID_LABELS
    if label not in judge_distribution.index
]

if missing_labels:
    print("\n⚠️ Judge limitation detected:")
    print(
        "The judge did not use these labels:",
        ", ".join(missing_labels)
    )

print("\nInterpretation:")
print(
    "The FLAN-T5-small judge produced valid labels, but its "
    "distribution differs substantially from the human labels. "
    "In particular, the judge did not assign any examples to "
    "'Acceptable', while human annotators used that category "
    "frequently. Therefore, the low agreement should be treated "
    "as evidence of judge-calibration/model-capacity limitations "
    "rather than as a reliable standalone measure of reply quality."
)

LLM-as-a-Judge Limitation Analysis

Judge audit size: 50
Human-LLM exact agreement: 26.00%
Cohen's kappa: 0.0154

LLM Judge distribution:
llm_judge_quality
Poor    29
Good    21
Name: count, dtype: int64

Human distribution:
human_reply_quality
Acceptable    25
Good          13
Poor          12
Name: count, dtype: int64

⚠️ Judge limitation detected:
The judge did not use these labels: Acceptable

Interpretation:
The FLAN-T5-small judge produced valid labels, but its distribution differs substantially from the human labels. In particular, the judge did not assign any examples to 'Acceptable', while human annotators used that category frequently. Therefore, the low agreement should be treated as evidence of judge-calibration/model-capacity limitations rather than as a reliable standalone measure of reply quality.


## 14. What is misleading about my headline number?

In [ ]:
# ============================================================
# ACTION ACCURACY ON GOLDEN SET
# ============================================================

from sklearn.metrics import accuracy_score

action_acc = accuracy_score(
    golden_scored["human_action"],
    golden_scored["decision"]
)

print(f"Human action agreement: {action_acc:.2%}")

print("\nHuman action distribution:")
print(golden_scored["human_action"].value_counts())

print("\nAgent decision distribution:")
print(golden_scored["decision"].value_counts())

Human action agreement: 75.00%

Human action distribution:
human_action
ESCALATE       150
AUTO_HANDLE     50
Name: count, dtype: int64

Agent decision distribution:
decision
ESCALATE    200
Name: count, dtype: int64


In [ ]:
# ============================================================
# WHAT IS MISLEADING ABOUT MY HEADLINE NUMBER?
# ============================================================

from sklearn.metrics import accuracy_score, f1_score

print("=" * 70)
print("WHAT IS MISLEADING ABOUT MY HEADLINE NUMBER?")
print("=" * 70)

# Calculate intent metrics directly from the scored golden dataset
headline_accuracy = accuracy_score(
    golden_scored["human_intent"],
    golden_scored["intent"]
)

headline_macro_f1 = f1_score(
    golden_scored["human_intent"],
    golden_scored["intent"],
    average="macro"
)

print("\nHeadline Intent Accuracy:", f"{headline_accuracy:.2%}")
print("Headline Intent Macro F1:", f"{headline_macro_f1:.4f}")

# Action accuracy
action_acc = accuracy_score(
    golden_scored["human_action"],
    golden_scored["decision"]
)

print("Action Accuracy:", f"{action_acc:.2%}")

# ============================================================
# INTERPRETATION
# ============================================================

print("\n" + "=" * 70)
print("WHY THE HEADLINE NUMBER CAN BE MISLEADING")
print("=" * 70)

print("""
1. Intent accuracy alone does not represent complete chatbot quality.
   The system also performs reply retrieval and escalation decisions.

2. The 27% intent accuracy is calculated on the 200-example human-labelled
   golden set, but performance is uneven across the nine intent classes.

3. Macro F1 is only 0.2896, showing that some intents are much harder
   for the classifier than others.

4. The agent escalated all 200 examples.
   This means the 75% action accuracy is partly caused by the fact that
   150 of the 200 human labels were ESCALATE.

5. The agent failed to AUTO_HANDLE the 50 examples labelled AUTO_HANDLE
   by humans. Therefore, 75% action accuracy should not be interpreted
   as a strong escalation policy.

6. The LLM judge also has limitations. It was tested on 50 examples,
   achieved only 26% exact agreement with human labels, and assigned
   zero examples to the Acceptable category.

Conclusion:
The headline intent accuracy should not be treated as a single measure
of overall chatbot quality. The results need to be considered together
with Macro F1, action agreement, failure cases, and LLM-judge agreement.
""")

print("=" * 70)
print("HEADLINE ANALYSIS COMPLETED")
print("=" * 70)

WHAT IS MISLEADING ABOUT MY HEADLINE NUMBER?

Headline Intent Accuracy: 27.00%
Headline Intent Macro F1: 0.2896
Action Accuracy: 75.00%

WHY THE HEADLINE NUMBER CAN BE MISLEADING

1. Intent accuracy alone does not represent complete chatbot quality.
   The system also performs reply retrieval and escalation decisions.

2. The 27% intent accuracy is calculated on the 200-example human-labelled
   golden set, but performance is uneven across the nine intent classes.

3. Macro F1 is only 0.2896, showing that some intents are much harder
   for the classifier than others.

4. The agent escalated all 200 examples.
   This means the 75% action accuracy is partly caused by the fact that
   150 of the 200 human labels were ESCALATE.

5. The agent failed to AUTO_HANDLE the 50 examples labelled AUTO_HANDLE
   by humans. Therefore, 75% action accuracy should not be interpreted
   as a strong escalation policy.

6. The LLM judge also has limitations. It was tested on 50 examples,
   achieved only 

## 15. Failure analysis — top 5 cases



In [ ]:
# ============================================================
# TOP 5 FAILURE MODES - SHORT
# ============================================================

print("=" * 60)
print("TOP 5 FAILURE MODES")
print("=" * 60)

failure_modes = [
    {
        "Failure Mode": "Over-prediction of 'other'",
        "Evidence": "92 incorrect 'other' predictions",
        "Hypothesis": "Weak separation between specific intents and 'other'"
    },
    {
        "Failure Mode": "Customer service confusion",
        "Evidence": "42 customer_service errors",
        "Hypothesis": "Overlapping customer-support language"
    },
    {
        "Failure Mode": "Product issue confusion",
        "Evidence": "26 product_issue errors",
        "Hypothesis": "Product complaints have varied wording"
    },
    {
        "Failure Mode": "Spelling / noisy input",
        "Evidence": "2/4 refund typo tests passed",
        "Hypothesis": "TF-IDF is sensitive to spelling variations"
    },
    {
        "Failure Mode": "Over-escalation",
        "Evidence": "0 AUTO_HANDLE; 200 ESCALATE",
        "Hypothesis": "Escalation policy is too conservative"
    }
]

failure_df = pd.DataFrame(failure_modes)

display(failure_df)

print("\nTop 5 failure analysis completed.")

TOP 5 FAILURE MODES


,Failure Mode,Evidence,Hypothesis
0,Over-prediction of 'other',92 incorrect 'other' predictions,Weak separation between specific intents and '...
1,Customer service confusion,42 customer_service errors,Overlapping customer-support language
2,Product issue confusion,26 product_issue errors,Product complaints have varied wording
3,Spelling / noisy input,2/4 refund typo tests passed,TF-IDF is sensitive to spelling variations
4,Over-escalation,0 AUTO_HANDLE; 200 ESCALATE,Escalation policy is too conservative



Top 5 failure analysis completed.


## 16. Decision log

Keep this concise: 10–15 non-obvious choices and why they were made.

In [ ]:
# ============================================================
# DECISION LOG
# ============================================================

decision_log = pd.DataFrame([
    [1, "Dataset", "AmazonHelp historical data",
     "Provides real customer-support conversations and resolutions."],

    [2, "Golden set size", "200 examples",
     "Provides a manageable human-labelled evaluation set."],

    [3, "Intent classes", "9 canonical intents",
     "Keeps the classification task consistent across the dataset."],

    [4, "Text representation", "TF-IDF",
     "Simple, fast and reproducible representation for text classification."],

    [5, "Classifier", "Logistic Regression",
     "Provides a strong and interpretable baseline for multi-class text classification."],

    [6, "Reply retrieval", "Cosine similarity",
     "Retrieves historically similar customer-support resolutions."],

    [7, "Supported auto-handle intents",
     "delivery, order, prime, return, product_issue",
     "Limits automatic handling to selected lower-risk support categories."],

    [8, "Safety threshold", "0.55 cosine similarity",
     "Avoids auto-handling weak historical matches; threshold should be validated on the golden set."],

    [9, "Sensitive intents",
     "refund_payment, account_login, customer_service",
     "These categories are treated conservatively because they may require human support."],

    [10, "Escalation policy", "Conservative escalation",
     "Reduces the risk of incorrect automatic responses."],

    [11, "Robustness testing", "Spelling variations",
     "Tests whether the classifier can handle noisy customer messages."],

    [12, "Baseline comparison",
     "Majority baseline + TF-IDF Logistic Regression",
     "Shows whether the proposed approach improves over simple baselines."],

    [13, "LLM judge", "FLAN-T5-small",
     "Provides a free open-source model for automated reply-quality evaluation."],

    [14, "Judge validation", "Human-LLM agreement",
     "Checks whether automated quality judgements agree with human labels."],

    [15, "Failure analysis", "Top 5 failure modes",
     "Identifies the main weaknesses and possible improvement areas."]
], columns=["#", "Decision", "Choice", "Reason"])

print("=" * 70)
print("DECISION LOG")
print("=" * 70)

display(decision_log)

print("\nTotal decisions:", len(decision_log))

DECISION LOG


,#,Decision,Choice,Reason
0,1,Dataset,AmazonHelp historical data,Provides real customer-support conversations a...
1,2,Golden set size,200 examples,Provides a manageable human-labelled evaluatio...
2,3,Intent classes,9 canonical intents,Keeps the classification task consistent acros...
3,4,Text representation,TF-IDF,"Simple, fast and reproducible representation f..."
4,5,Classifier,Logistic Regression,Provides a strong and interpretable baseline f...
5,6,Reply retrieval,Cosine similarity,Retrieves historically similar customer-suppor...
6,7,Supported auto-handle intents,"delivery, order, prime, return, product_issue",Limits automatic handling to selected lower-ri...
7,8,Safety threshold,0.55 cosine similarity,Avoids auto-handling weak historical matches; ...
8,9,Sensitive intents,"refund_payment, account_login, customer_service",These categories are treated conservatively be...
9,10,Escalation policy,Conservative escalation,Reduces the risk of incorrect automatic respon...



Total decisions: 15


In [ ]:
# ============================================================
# FINAL EXPORT - EVALUATION ARTIFACTS
# ============================================================

# Do not commit API keys.

# 1. Golden evaluation
golden_scored.to_csv(
    "/content/amazonhelp_golden_scored.csv",
    index=False
)

# 2. Baseline comparison
baseline_comparison.to_csv(
    "/content/amazonhelp_baseline_comparison.csv",
    index=False
)

# 3. Top 5 failure summary
failure_df.to_csv(
    "/content/amazonhelp_top5_failure_cases.csv",
    index=False
)

# 4. Decision log
decision_log.to_csv(
    "/content/amazonhelp_decision_log.csv",
    index=False
)

# 5. LLM judge audit
judge_audit.to_csv(
    "/content/amazonhelp_llm_judge_audit_50.csv",
    index=False
)

print("=" * 60)
print("FINAL EVALUATION ARTIFACTS SAVED")
print("=" * 60)

files = [
    "/content/amazonhelp_golden_scored.csv",
    "/content/amazonhelp_baseline_comparison.csv",
    "/content/amazonhelp_top5_failure_cases.csv",
    "/content/amazonhelp_decision_log.csv",
    "/content/amazonhelp_llm_judge_audit_50.csv"
]

for p in files:
    print("✅", p)

print("\nTotal files exported:", len(files))


FINAL EVALUATION ARTIFACTS SAVED
✅ /content/amazonhelp_golden_scored.csv
✅ /content/amazonhelp_baseline_comparison.csv
✅ /content/amazonhelp_top5_failure_cases.csv
✅ /content/amazonhelp_decision_log.csv
✅ /content/amazonhelp_llm_judge_audit_50.csv

Total files exported: 5
